# ЛР 1. Первинний аналіз даних
**Курс:** Технологія блокчейн та машинне навчання · **Група:** ПЗПІ-23-5
**Тема 8:** Реєстрація авторства текстів і виявлення плагіату
**Репозиторій:** <посилання на репозиторій>

## Склад команди та внесок
| Учасник | Внесок у цей ноутбук |
|---|---|
| Буткевич А. В. | <заповнити> |
| Тохтамиш М. С. | <заповнити> |

## Мета роботи
Дослідити набір даних для моделі виявлення плагіату: описати його, провести первинний аналіз, виявити проблеми та обрати схему валідації й метрики якості. Ці рішення ми використаємо в ЛР 2–4.

## Зміст
1. Опис набору даних (джерело, ліцензія, розмір, словник ознак)
2. Пропуски, дублікати, викиди
3. Розподіл ознак і баланс класів
4. Кореляції та перевірка на витік
5. Висновки з аналізу
6. Схема поділу train/test і метрики
7. Самоперевірка

## Нумерація та посилання
Рисунки й таблиці мають наскрізну нумерацію (Рис. 1, Табл. 1...), підписи та згадування в тексті. Ті самі номери використовуємо у звіті.

## Використання ШІ-асистентів
<Вкажіть, для чого саме, наприклад: «Claude допомагав із каркасом ноутбука та кодом графіків; висновки й вибір метрик зроблено командою».>

In [ ]:
# Залежності: pip install pandas numpy matplotlib seaborn scikit-learn datasets
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, average_precision_score, roc_auc_score, precision_score, recall_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## 1. Опис набору даних

Завдання моделі: для пари текстів (`text_a`, `text_b`) визначити, чи є `text_b` плагіатом/перефразуванням `text_a` (`label = 1`) чи ні (`label = 0`).

**За замовчуванням використано MRPC (GLUE).** Якщо ваша команда обрала інший набір (PAN, PAWS, Quora Question Pairs або власний), змініть `DATASET` і заповніть таблицю нижче.

| Параметр | Значення |
|---|---|
| Назва | MRPC (Microsoft Research Paraphrase Corpus), у складі GLUE |
| Джерело | https://huggingface.co/datasets/nyu-mll/glue (конфігурація `mrpc`) |
| Ліцензія | <перевірити та вказати ліцензію на сторінці набору> |
| Мова | англійська |
| Розмір | див. нижче, Табл. 1 |

Якщо завантажити набір не вдалося (немає мережі), ноутбук згенерує **синтетичний** набір пар, щоб увесь аналіз можна було запустити. Тоді у звіті слід чесно вказати, що дані синтетичні.

In [ ]:
DATASET = "mrpc"   # змініть, якщо обрано інший набір

def load_mrpc():
    from datasets import load_dataset
    ds = load_dataset("glue", "mrpc")
    parts = []
    for split in ds:
        d = ds[split].to_pandas()
        d["orig_split"] = split
        parts.append(d)
    d = pd.concat(parts, ignore_index=True)
    return d.rename(columns={"sentence1": "text_a", "sentence2": "text_b"})[["text_a", "text_b", "label", "orig_split"]]

def make_synthetic(n=3000):
    """Резервний варіант: синтетичні пари (тільки для перевірки коду)."""
    vocab = [f"word{i}" for i in range(800)]
    rows = []
    for _ in range(n):
        a = list(rng.choice(vocab, size=int(rng.integers(8, 40))))
        if rng.random() < 0.35:   # плагіат: заміна частини слів
            b = [w if rng.random() > 0.25 else rng.choice(vocab) for w in a]
            label = 1
        else:
            b = list(rng.choice(vocab, size=int(rng.integers(8, 40))))
            label = 0
        rows.append((" ".join(a), " ".join(b), label, "synthetic"))
    return pd.DataFrame(rows, columns=["text_a", "text_b", "label", "orig_split"])

try:
    df = load_mrpc()
    SOURCE = "MRPC (GLUE)"
except Exception as e:
    print("Не вдалося завантажити набір:", repr(e)[:200])
    df = make_synthetic()
    SOURCE = "СИНТЕТИЧНІ ДАНІ"
print("Джерело:", SOURCE)
df.head()

In [ ]:
summary = pd.DataFrame({
    "Показник": ["Кількість пар", "Унікальних text_a", "Унікальних text_b", "Частка класу 1 (плагіат)", "Кількість ознак (колонок)"],
    "Значення": [len(df), df.text_a.nunique(), df.text_b.nunique(), round(df.label.mean(), 3), df.shape[1]],
})
print("Табл. 1. Загальні характеристики набору")
summary

**Табл. 2. Словник ознак**

| Колонка | Тип | Опис |
|---|---|---|
| `text_a` | текст | Оригінальний текст |
| `text_b` | текст | Текст-кандидат на плагіат |
| `label` | 0/1 | Цільова змінна: 1 - перефразування/плагіат |
| `orig_split` | категорія | Початковий поділ джерела (train/validation/test); як ознаку не використовуємо |
| `len_a`, `len_b` | число | Довжина текстів у словах (обчислюється нижче) |
| `len_ratio` | число | `min(len_a, len_b) / max(len_a, len_b)` |
| `jaccard` | число | Схожість множин слів |
| `tfidf_cos` | число | Косинусна схожість TF-IDF векторів |

## 2. Пропуски, дублікати, викиди

In [ ]:
df["len_a"] = df.text_a.fillna("").str.split().str.len()
df["len_b"] = df.text_b.fillna("").str.split().str.len()

missing = df[["text_a", "text_b", "label"]].isna().sum()
empty = pd.Series({"text_a": (df.text_a.fillna("").str.strip() == "").sum(),
                   "text_b": (df.text_b.fillna("").str.strip() == "").sum()})
dups_pair = df.duplicated(["text_a", "text_b"]).sum()
dups_a = df.duplicated("text_a").sum()
print("Пропуски:\n", missing.to_string())
print("\nПорожні рядки:\n", empty.to_string())
print(f"\nДублікатів пар: {dups_pair}; повторів text_a: {dups_a}")

fig, ax = plt.subplots(figsize=(5, 3))
(missing + empty.reindex(missing.index).fillna(0)).plot.bar(ax=ax, color="#4c72b0")
ax.set_xlabel("Колонка"); ax.set_ylabel("Кількість пропусків/порожніх")
ax.set_title("Рис. 1. Пропуски та порожні значення")
plt.tight_layout(); plt.show()

**Висновок до Рис. 1:** <опишіть, чи є пропуски та порожні тексти і що з ними робити (видалити / заповнити).>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
sns.boxplot(data=df[["len_a", "len_b"]], ax=axes[0], palette="Set2")
axes[0].set_xlabel("Текст"); axes[0].set_ylabel("Довжина, слів")
axes[0].set_title("Рис. 2. Викиди за довжиною (boxplot)")

q1, q3 = df.len_b.quantile([.25, .75]); iqr = q3 - q1
outl = ((df.len_b < q1 - 1.5 * iqr) | (df.len_b > q3 + 1.5 * iqr))
sns.histplot(df.len_b, bins=40, ax=axes[1], color="#55a868")
axes[1].set_xlabel("Довжина text_b, слів"); axes[1].set_ylabel("Кількість пар")
axes[1].set_title(f"Розподіл довжини (викидів за IQR: {outl.sum()})")
plt.tight_layout(); plt.show()

**Висновок до Рис. 2:** <чи є дуже короткі/довгі тексти, як обмежимо довжину при токенізації.>

## 3. Розподіл ознак і баланс класів

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
cnt = df.label.value_counts().sort_index()
ax.bar(["0 (оригінал)", "1 (плагіат)"], cnt.values, color=["#4c72b0", "#c44e52"])
for i, v in enumerate(cnt.values):
    ax.text(i, v, f"{v} ({v/len(df):.0%})", ha="center", va="bottom")
ax.set_xlabel("Клас"); ax.set_ylabel("Кількість пар")
ax.set_title("Рис. 3. Баланс класів")
plt.tight_layout(); plt.show()

**Висновок до Рис. 3:** <наскільки незбалансовані класи; чи потрібні class_weight / стратифікація; чому accuracy недостатньо.>

In [ ]:
def tokset(s):
    return set(re.findall(r"\w+", str(s).lower()))

def jaccard(a, b):
    A, B = tokset(a), tokset(b)
    return len(A & B) / len(A | B) if (A | B) else 0.0

df["len_ratio"] = np.minimum(df.len_a, df.len_b) / np.maximum(np.maximum(df.len_a, df.len_b), 1)
df["jaccard"] = [jaccard(a, b) for a, b in zip(df.text_a, df.text_b)]

tfidf = TfidfVectorizer().fit(pd.concat([df.text_a, df.text_b]).fillna(""))
A = tfidf.transform(df.text_a.fillna("")); B = tfidf.transform(df.text_b.fillna(""))
df["tfidf_cos"] = np.asarray(A.multiply(B).sum(axis=1)).ravel()   # вектори L2-нормовані

FEATURES = ["len_a", "len_b", "len_ratio", "jaccard", "tfidf_cos"]
df[FEATURES].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, col, xl in zip(axes, ["jaccard", "tfidf_cos", "len_ratio"],
                       ["Jaccard", "Косинус TF-IDF", "Відношення довжин"]):
    sns.histplot(data=df, x=col, hue="label", bins=30, stat="density", common_norm=False,
                 ax=ax, palette=["#4c72b0", "#c44e52"])
    ax.set_xlabel(xl); ax.set_ylabel("Щільність")
axes[1].set_title("Рис. 4. Розподіл ознак схожості за класами")
plt.tight_layout(); plt.show()

**Висновок до Рис. 4:** <наскільки ознаки розділяють класи; чи перетинаються розподіли; чи достатньо лексичної схожості, чи потрібна семантична модель (embeddings).>

## 4. Кореляції та перевірка на витік

In [ ]:
corr = df[FEATURES + ["label"]].corr()
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Рис. 5. Кореляційна матриця ознак і цільової змінної")
plt.tight_layout(); plt.show()

**Висновок до Рис. 5:** <які ознаки сильно корелюють між собою (мультиколінеарність) і з `label`.>

In [ ]:
# Перевірка на витік 1: AUC окремих ознак (підозріло, якщо одна ознака майже ідеально передбачає label)
auc_rows = []
for f in FEATURES:
    auc = roc_auc_score(df.label, df[f])
    auc_rows.append((f, round(max(auc, 1 - auc), 3)))
leak_tbl = pd.DataFrame(auc_rows, columns=["Ознака", "AUC однієї ознаки"]).sort_values("AUC однієї ознаки", ascending=False)
print("Табл. 3. Якість окремих ознак (AUC)")
display(leak_tbl)

# Перевірка на витік 2: чи повторюється той самий text_a у різних частинах
if "orig_split" in df:
    sp = df.groupby("text_a").orig_split.nunique()
    print("text_a, що зустрічаються у кількох початкових частинах:", int((sp > 1).sum()))

**Висновок до Табл. 3:** <чи є ознака-«підглядач» (AUC близько 1.0) і що робимо; чи є повтори text_a між частинами.>

## 5. Висновки з аналізу

| № | Проблема в даних | Доказ (Рис./Табл.) | Як розв'язуватимемо |
|---|---|---|---|
| 1 | <наприклад, дисбаланс класів> | Рис. 3 | <class_weight, PR-AUC, стратифікація> |
| 2 | <дублікати / повтори text_a> | розділ 2 | <групований поділ, видалення дублікатів> |
| 3 | <викиди за довжиною> | Рис. 2 | <обрізання довжини / truncation> |
| 4 | <лексична схожість недостатня> | Рис. 4 | <семантичні embeddings, fine-tuning> |
| 5 | <ризик витоку> | Табл. 3 | <контроль ознак, групований split> |

## 6. Схема поділу даних і метрики

**Схема валідації.**
- Поділ 80/20 на train/test **з групуванням за `text_a`**: той самий оригінал не може бути одночасно в train і в test (інакше модель «запам'ятає» текст, а метрики завищаться).
- Усередині train - GroupKFold (5 фолдів) для підбору гіперпараметрів; test використовуємо один раз, наприкінці.
- Фіксуємо `random_state = 42` для відтворюваності.

**Метрики.**
- **F1 (клас «плагіат»)** - основна, бо баланс класів неідеальний, а важливі і хибні спрацьовування, і пропуски.
- **Precision та Recall** окремо: хибне звинувачення в плагіаті коштує інакше, ніж пропуск.
- **PR-AUC** - поріг-незалежна метрика для незбалансованих класів; **ROC-AUC** - для порівняння з літературою.
- Цільове значення з нефункціональних вимог: <наприклад, F1 ≥ 0.85>.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, df.label, groups=df.text_a))
train, test = df.iloc[train_idx], df.iloc[test_idx]

overlap = set(train.text_a) & set(test.text_a)
print(f"train: {len(train)}, test: {len(test)}; спільних text_a: {len(overlap)} (має бути 0)")
print("Частка класу 1: train={:.3f}, test={:.3f}".format(train.label.mean(), test.label.mean()))

### (Додатково) Перша базова модель

In [ ]:
clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
clf.fit(train[FEATURES], train.label)
proba = clf.predict_proba(test[FEATURES])[:, 1]
pred = (proba >= 0.5).astype(int)

res = pd.DataFrame({
    "Метрика": ["F1", "Precision", "Recall", "PR-AUC", "ROC-AUC"],
    "Значення": [f1_score(test.label, pred), precision_score(test.label, pred), recall_score(test.label, pred),
                 average_precision_score(test.label, proba), roc_auc_score(test.label, proba)],
}).round(3)
print("Табл. 4. Базова модель (логістична регресія на ознаках схожості)")
res

## 7. Самоперевірка (розділ «Дані» переліку ЛР 1)

| Пункт переліку | Виконано | Де в ноутбуці |
|---|---|---|
| Опис набору: джерело, ліцензія, розмір, словник ознак | <так/ні> | Розділ 1, Табл. 1-2 |
| Первинний аналіз, не менше 6 графіків | <так/ні> | Рис. 1-5 + Рис. 6 (<додайте ще один, напр. довжини за класами>) |
| Висновки з аналізу: проблеми й розв'язки | <так/ні> | Розділ 5 |
| Метрики якості та схема валідації з обґрунтуванням | <так/ні> | Розділ 6 |
| Кожен графік має підписи осей і висновок | <так/ні> | Розділи 2-4 |